# Using Dynamic Beta for Portfolio Hedging

**Hedging** means reducing your risk by taking an offsetting position.

If you own a stock and worry the overall market might drop, you can *short* an index
(like SPY) to offset potential losses. The key question is: **how much** of the index
should you short?

That is exactly what beta tells you. If your stock has a beta of 1.3, it moves 1.3x
as much as the market -- so you need to short 1.3x your position value in SPY to be
fully hedged.

In [ ]:
!pip install grubeta[data] -q

In [ ]:
from grubeta import estimate_beta

## Estimate Beta for Your Stock

We will estimate the dynamic beta for AAPL relative to SPY.

In [ ]:
result = estimate_beta("AAPL", "SPY")
print(result["summary"])

## Calculate the Hedge Ratio

The hedge ratio formula is straightforward:

$$\text{Hedge Notional} = \beta \times \text{Position Value}$$

If you hold $100,000 worth of AAPL and the current beta is 1.2, you would short
$120,000 worth of SPY to neutralize market risk.

To convert that into shares:

$$\text{Shares to Short} = \frac{\text{Hedge Notional}}{\text{SPY Price}}$$

In [ ]:
# Get the most recent beta value
current_beta = result["beta"].dropna().iloc[-1]

# Portfolio parameters
position_value = 100_000  # $100K in AAPL
spy_price = 450  # Approximate SPY price -- update as needed

# Calculate hedge
hedge_notional = current_beta * position_value
shares_to_short = hedge_notional / spy_price

print(f"Current AAPL beta:     {current_beta:.4f}")
print(f"Position value:        ${position_value:,.0f}")
print(f"Hedge notional:        ${hedge_notional:,.0f}")
print(f"SPY shares to short:   {shares_to_short:.1f}")

## Compare Multiple Stocks' Hedge Requirements

Different stocks have different betas, so the hedge amount varies.
Let's compare several well-known names.

In [ ]:
tickers = ["AAPL", "MSFT", "JNJ", "XOM", "JPM"]
position_value = 100_000
spy_price = 450  # Approximate SPY price -- update as needed

print(f"{'Ticker':<8} {'Beta':>8} {'Hedge ($)':>12} {'SPY Shares':>12}")
print("-" * 44)

for ticker in tickers:
    res = estimate_beta(ticker, "SPY", plot=False, verbose=False)
    beta = res["beta"].dropna().iloc[-1]
    hedge = beta * position_value
    shares = hedge / spy_price
    print(f"{ticker:<8} {beta:>8.4f} {hedge:>12,.0f} {shares:>12.1f}")

## Key Takeaways

- **High-beta stocks** (e.g., tech) require *more* hedging because they amplify market moves.
- **Low-beta stocks** (e.g., consumer staples, utilities) require *less* hedging.
- **Dynamic beta matters**: A static hedge set months ago may be stale. GRUBeta's time-varying estimate lets you adjust your hedge as conditions change.

**When hedging helps**: You are worried about a broad market decline but want to keep your individual stock positions.

**When hedging hurts**: If the market rallies, your short SPY position creates a drag. Hedging removes upside along with the downside -- that is the cost of insurance.